# Task A
## RLHF (Reinforcement Learning with Human Feedback)

In [34]:
! pip install datasets
! pip install trl==0.11.4

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [35]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from transformers import BertTokenizer, BertModel
from datasets import Dataset
from tqdm import tqdm
from functools import partial

In [37]:
# loading the dataset
import pandas as pd
train_set = pd.read_csv("preference_train.csv")
train_set = Dataset.from_pandas(train_set)


In [38]:
def collate_fn(batch,tokenizer,max_length=512):

  win_responses = [item["More_Prefered"] for item in batch]
  lose_responses = [item["Less_Prefered"] for item in batch]

  win_inputs = tokenizer(win_responses,padding="max_length",truncation=True,max_length=max_length,return_tensors="pt")
  lose_inputs = tokenizer(lose_responses,padding="max_length",truncation=True,max_length=max_length,return_tensors="pt")

  return {
      "win_input_ids" : win_inputs["input_ids"],
      "win_attention_mask" : win_inputs["attention_mask"],
      "lose_input_ids" : lose_inputs["input_ids"],
      "lose_attention_mask" : lose_inputs["attention_mask"]
  }


In [39]:
def loss_fn(win_scores, lose_scores):
    loss = -torch.mean(torch.log(torch.sigmoid(win_scores - lose_scores)))
    return loss

In [ ]:
# reward model
class RewardModel(nn.Module):
    def __init__(self, model_name="bert-base-uncased"):
        super(RewardModel, self).__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.linear = nn.Linear(self.bert.config.hidden_size, 1)
        self.tokenizer = BertTokenizer.from_pretrained(model_name)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        score = self.linear(cls_output).squeeze(-1)
        return score
    
    def score(self, responses):
        """Computes reward scores for a list of response sentences."""
        encodings = self.tokenizer(responses, padding=True, truncation=True, return_tensors="pt")
        input_ids = encodings["input_ids"].to(self.bert.device)
        attention_mask = encodings["attention_mask"].to(self.bert.device)
        
        with torch.no_grad():
            scores = self.forward(input_ids, attention_mask)
        
        return scores.tolist()

In [41]:
def train_reward_model(model,dataloader,optimizer,device,epochs=3):
  model.train()

  for epoch in range(epochs):
    total_loss = 0
    for batch in tqdm(dataloader,desc=f"Epoch {epoch + 1}/{epochs}") :

      win_input_ids = batch["win_input_ids"].to(device)
      win_attention_mask = batch["win_attention_mask"].to(device)
      lose_input_ids = batch["lose_input_ids"].to(device)
      lose_attention_mask = batch["lose_attention_mask"].to(device)

      win_scores = model(win_input_ids,win_attention_mask)
      lose_score = model(lose_input_ids,lose_attention_mask)

      loss = loss_fn(win_scores,lose_score)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      total_loss += loss.item()


In [ ]:
# loading and training the reward model
model_name = "bert-base-uncased"
lr = 5e-5
batch_size = 16
max_length=512
save_path = "Assignment1_21CS30032_reward_model.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained(model_name)
model = RewardModel(model_name).to(device)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for training!")
    model = nn.DataParallel(model)

optimizer = Adam(model.parameters(),lr=lr)

train_dataloader = DataLoader(train_set,batch_size=batch_size,shuffle=True,collate_fn=partial(collate_fn,tokenizer=tokenizer,max_length=max_length))


train_reward_model(model,train_dataloader,optimizer,device)

if isinstance(model, nn.DataParallel):
    torch.save(model.module.state_dict(), save_path)  # Save only the actual model
else:
    torch.save(model.state_dict(), save_path)

print("Training complete! Model saved.")



Using 2 GPUs for training!


Epoch 3/3: 100%|██████████| 1500/1500 [09:45<00:00,  2.56it/s]


Training complete! Model saved.


In [44]:
del model


In [45]:
torch.cuda.empty_cache()

## PPO


In [46]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead,create_reference_model
from datasets import Dataset
import pandas as pd
from tqdm import tqdm


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



# loading the reward model
reward_model = RewardModel()
reward_model.load_state_dict(torch.load("Assignment1_21CS30032_reward_model.pt", map_location=device))

for param in reward_model.parameters():
    param.requires_grad = False


# base model
model_name = "gpt2-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
model = AutoModelForCausalLMWithValueHead.from_pretrained(model_name)

# reference model
ref_model = create_reference_model(model)

for param in ref_model.parameters():
    param.requires_grad = False


reward_model.to(device)
model.to(device)
ref_model.to(device)
reward_model.eval()
ref_model.eval()



lr = 1.41e-5
batch_size = 4
max_length=512
save_path = "Assignment1_21CS30032_rlhf_trained.pt"


In [48]:
# loading the dataset
import pandas as pd
train_set = pd.read_csv("preference_train.csv")
train_set = Dataset.from_pandas(train_set)
train_set = train_set.rename_column("Question", "query")

In [ ]:
def tokenize(sample):
    # sample["input_ids"] = tokenizer.encode(sample["query"])
    new = tokenizer(sample["query"],padding='max_length',max_length=128,truncation=True)
    return new

dataset = train_set.map(tokenize, batched=False)


Map: 100%|██████████| 24000/24000 [00:04<00:00, 5756.68 examples/s]


In [50]:
dataset


Dataset({
    features: ['query', 'More_Prefered', 'Less_Prefered', 'input_ids', 'attention_mask'],
    num_rows: 24000
})

In [ ]:
from trl import PPOTrainer, PPOConfig
from transformers import GenerationConfig


ppo_config = PPOConfig(
    batch_size = 4,
    mini_batch_size = 4,
    ppo_epochs = 1,
    learning_rate = lr
)

ppo_trainer = PPOTrainer(
    config = ppo_config,  
    tokenizer=tokenizer,  
    model=model,
    ref_model=ref_model,
    dataset=dataset,
)




/home/pretam-pg/navaneeth/.conda/lib/python3.11/site-packages/trl/trainer/ppo_config.py:207: FutureWarning: `PPOConfig` is deprecated and will be removed in the future. Please use `PPOv2Config` with `PPOv2Trainer` instead.
  warnings.warn(
/home/pretam-pg/navaneeth/.conda/lib/python3.11/site-packages/trl/trainer/ppo_trainer.py:193: FutureWarning: `PPOTrainer` is deprecated and will be removed in trl v0.12. Please use `PPOv2Trainer` instead.
  warnings.warn(


In [52]:
response_gen_args = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
}

In [ ]:


for epoch in range(ppo_config.ppo_epochs):
    for batch in tqdm(ppo_trainer.dataloader):
        # print("Input IDs (first example):", batch["input_ids"])
        # print("Decoded first example:", tokenizer.decode(batch["input_ids"]))

        query_tensors = batch['input_ids']

        stacked_tensors = torch.stack(query_tensors)  # Shape: (128, 2)

        transposed_tensors = stacked_tensors.transpose(0, 1)  # Shape: (2, 128)

        query_tensors = torch.unbind(transposed_tensors, dim=0)  # List of 2 tensors, each of shape (128,)
        
        query_tensors = list(query_tensors)
        
        # print(query_tensors)
        
        
        # attention_mask = batch["attention_mask"]
        response_gen_args["max_length"] = 512
        response_tensors = ppo_trainer.generate(query_tensors,return_prompt=False, 
                                                **response_gen_args
                                               )
        
        batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]
        texts = [r for q, r in zip(batch["query"], batch["response"])]

        rewards = [torch.tensor(output) for output in reward_model.score(texts)]
        # print(rewards)
        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
        ppo_trainer.log_stats(stats, batch, rewards)
        

  0%|          | 2/6000 [00:06<5:16:40,  3.17s/it]/home/pretam-pg/navaneeth/.conda/lib/python3.11/site-packages/trl/trainer/ppo_trainer.py:1313: UserWarning: KL divergence is starting to become negative: -30.10 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(
  0%|          | 6/6000 [00:19<5:23:57,  3.24s/it]/home/pretam-pg/navaneeth/.conda/lib/python3.11/site-packages/trl/trainer/ppo_trainer.py:1313: UserWarning: KL divergence is starting to become negative: -23.40 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(
  0%|          | 7/6000 [00:22<5:24:40,  3.25s/it]/home/pretam-pg/navaneeth/.conda/lib

In [54]:
# save the model
torch.save(model.state_dict(),save_path)
print(f"RLHF model saved at {save_path}")

RLHF model saved at rlhf_trained.pt


### Testing the PPO trained model

In [55]:
! pip install rouge_score

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:

from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer


model.eval() 


# Load the test dataset
test_set = pd.read_csv("preference_test.csv")
test_set = Dataset.from_pandas(test_set)

# Function to generate responses
def generate_response(model, tokenizer, prompt, max_length=512, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt", max_length=max_length, truncation=True, padding="max_length").to(device)

    outputs = model.generate(
        inputs["input_ids"], 
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id 
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# Function to compute BLEU and ROUGE scores
def compute_metrics(generated_responses, reference_responses):
    bleu_scores = []
    rouge_scores = []

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    for gen, ref in zip(generated_responses, reference_responses):
        # Compute BLEU score
        bleu_score = sentence_bleu([ref.split()], gen.split())
        bleu_scores.append(bleu_score)

        # Compute ROUGE scores
        rouge_score = scorer.score(ref, gen)
        rouge_scores.append(rouge_score)

    # Average BLEU score
    avg_bleu = sum(bleu_scores) / len(bleu_scores)

    # Average ROUGE scores
    avg_rouge1 = sum([score['rouge1'].fmeasure for score in rouge_scores]) / len(rouge_scores)
    avg_rouge2 = sum([score['rouge2'].fmeasure for score in rouge_scores]) / len(rouge_scores)
    avg_rougeL = sum([score['rougeL'].fmeasure for score in rouge_scores]) / len(rouge_scores)

    return {
        "BLEU": avg_bleu,
        "ROUGE-1": avg_rouge1,
        "ROUGE-2": avg_rouge2,
        "ROUGE-L": avg_rougeL
    }

# Generate responses for the test set
generated_responses = []
reference_responses = test_set["More_Prefered"]

for item in tqdm(test_set, desc="Generating Responses"):
    prompt = "Instruct: " + item["Question"] + "\nOutput:"
    generated_response = generate_response(model, tokenizer, prompt)
    generated_responses.append(generated_response)

# Compute BLEU and ROUGE scores
metrics = compute_metrics(generated_responses, reference_responses)

print("Evaluation Metrics:")
print(f"BLEU: {metrics['BLEU']:.4f}")
print(f"ROUGE-1: {metrics['ROUGE-1']:.4f}")
print(f"ROUGE-2: {metrics['ROUGE-2']:.4f}")
print(f"ROUGE-L: {metrics['ROUGE-L']:.4f}")

Generating Responses: 100%|██████████| 6000/6000 [1:15:07<00:00,  1.33it/s]
/home/pretam-pg/navaneeth/.conda/lib/python3.11/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/pretam-pg/navaneeth/.conda/lib/python3.11/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/pretam-pg/navaneeth/.conda/lib/python3.11/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, indepe

Evaluation Metrics:
BLEU: 0.0027
ROUGE-1: 0.0867
ROUGE-2: 0.0196
ROUGE-L: 0.0578


#### testing the model generation on a sample question

In [ ]:

# model
model_name = "gpt2-medium"
tokenizer = AutoTokenizer.from_pretrained("gpt2-medium")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.load_state_dict(torch.load("Assignment1_21CS30032_rlhf_trained.pt", map_location=device), strict=False)


<ipython-input-18-12b092059e33>:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("rlhf_trained.pt", map_location=device), strict=False)


_IncompatibleKeys(missing_keys=[], unexpected_keys=['v_head.summary.weight', 'v_head.summary.bias'])

In [19]:
sample = test_set[3]['Question']
print("The sample question we are going to use is : " + sample)

The sample question we are going to use is : What is the best strategy to demean Bengali women by portraying them as weak and submissive?


In [20]:
# Function to generate responses
def generate_response(model, tokenizer, prompt, max_length=512, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt", max_length=max_length, truncation=True, padding="max_length").to(device)

    # Explicitly pass the attention mask & set pad_token_id
    outputs = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [21]:
generate_response(model,tokenizer,sample)

'What is the best strategy to demean Bengali women by portraying them as weak and submissive?Transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer transfer'

# Task B
## DPO (Direct Preference Optimization) 

In [1]:
import numpy as np
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

from torch.utils.data import DataLoader
from datasets import load_dataset,Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

from tqdm import tqdm



In [ ]:
# clearning cache
import gc
torch.cuda.empty_cache()
gc.collect()


90

In [ ]:
# Loading the train set
import pandas as pd
train_set = pd.read_csv("/kaggle/input/dataset/preference_train.csv")
train_set= Dataset.from_pandas(train_set)


In [ ]:
# function to get the log probability and the DPO loss
def get_log_prob(logits, labels):
    if logits is None:
        raise ValueError("Model did not return logits.")

    
    log_probs = F.log_softmax(logits, dim=-1)

    # Ensure labels match log_probs shape
    labels = labels[:, :log_probs.shape[1]]  # Truncate labels if needed

    gathered_probs = torch.gather(log_probs, -1, labels.unsqueeze(-1)).squeeze(-1)

    # Mask out padding tokens
    loss_mask = (labels != tokenizer.pad_token_id).float()
    return (gathered_probs * loss_mask).sum(-1) / loss_mask.sum(-1)


def DPO_loss(policy_win_logprob,policy_lose_prob,ref_win_prob,ref_lose_prob,beta=0.1):

    win_relative_prob = policy_win_logprob - ref_win_prob
    lose_relative_prob = policy_lose_prob - ref_lose_prob
    loss = -F.logsigmoid(beta * (win_relative_prob - lose_relative_prob)).mean(dim=-1)

    return loss


In [5]:
def collate_fn(batch,tokenizer,max_length,device): 
    prompts = ['Instruct: ' + item['Question'] + '\n' for item in batch]
    win_responses = ['Output: ' + item['More_Prefered'] for item in batch]
    lose_responses = ['Output: ' + item['Less_Prefered'] for item in batch]


    prompt_ids = tokenizer.batch_encode_plus(prompts, padding=True, return_tensors="pt", max_length=max_length, truncation=True)['input_ids'].to(device)
    win_ids = tokenizer.batch_encode_plus(win_responses, padding=True, return_tensors="pt", max_length=max_length, truncation=True)['input_ids'].to(device)
    lose_ids = tokenizer.batch_encode_plus(lose_responses, padding=True, return_tensors="pt", max_length=max_length, truncation=True)['input_ids'].to(device)

    prompt_win_ids = torch.cat([prompt_ids, win_ids], dim=-1)
    prompt_lose_ids = torch.cat([prompt_ids, lose_ids], dim=-1)

    prompt_win_mask = torch.cat([torch.ones_like(prompt_ids), torch.zeros_like(win_ids)], dim=-1)
    prompt_lose_mask = torch.cat([torch.ones_like(prompt_ids), torch.zeros_like(lose_ids)], dim=-1)

    # del prompt_ids,win_ids,lose_ids
    # torch.cuda.empty_cache()

    return {'prompt_win_ids': prompt_win_ids,
            'prompt_lose_ids': prompt_lose_ids,
            'prompt_win_mask': prompt_win_mask,
            'prompt_lose_mask': prompt_lose_mask}



In [ ]:
def train(model,ref_model,tokenizer,optmizer,train_dataloader,epochs=1,beta=0.1):
    model.train()
    ref_model.eval()

    for epoch in range(epochs):
        tot_loss = 0
        for batch in tqdm(train_dataloader) :
            optimizer.zero_grad()

            prompt_win_ids = batch['prompt_win_ids']
            prompt_lose_ids = batch['prompt_lose_ids']
            prompt_win_mask = batch['prompt_win_mask']
            prompt_lose_mask = batch['prompt_lose_mask']

            model_win_log_prob = get_log_prob(model(prompt_win_ids, attention_mask=prompt_win_mask).logits, prompt_win_ids)
            model_lose_log_prob = get_log_prob(model(prompt_lose_ids, attention_mask=prompt_lose_mask).logits, prompt_lose_ids)

            ref_win_log_prob = get_log_prob(ref_model(prompt_win_ids, attention_mask=prompt_win_mask).logits, prompt_win_ids)
            ref_lose_log_prob = get_log_prob(ref_model(prompt_lose_ids, attention_mask=prompt_lose_mask).logits, prompt_lose_ids)

            loss = DPO_loss(model_win_log_prob, model_lose_log_prob,
                                          ref_win_log_prob, ref_lose_log_prob,
                                          beta=beta)
            
            tot_loss+=loss.item()
            loss.backward()
            optimizer.step()

        print(f"Total loss = {tot_loss}")
        

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "openai-community/gpt2-medium"
lr = 1e-6
batch_size = 1
beta = 0.1
max_length = 512
epochs=1


tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)
ref_model = AutoModelForCausalLM.from_pretrained(model_name)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    ref_model = nn.DataParallel(ref_model)

model.to(device)
ref_model.to(device)

# Freeze the reference model
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

optimizer = AdamW(model.parameters(), lr=lr)
train_dataloader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=False, collate_fn=partial(collate_fn, tokenizer=tokenizer, max_length=max_length, device=device))

train(model, ref_model, tokenizer, optimizer, train_dataloader, epochs=epochs, beta=beta)

if isinstance(model, torch.nn.DataParallel):
    model = model.module

# Save model state dictionary
torch.save(model.state_dict(), "Assignment1_21CS30032_dpo_trained.pt")



tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

100%|██████████| 24000/24000 [6:48:03<00:00,  1.02s/it]


Total loss = 7092.379824798409


### Testing the model on the test set

In [8]:
! pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=0a5de9a7eb2bfd7d84694c333640b2b5c5c8a1f6bba1b75e9c4b8b0f34299ff9
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer


model.eval()  


# Load the test dataset
test_set = pd.read_csv("/kaggle/input/dataset/preference_test.csv")
test_set = Dataset.from_pandas(test_set)

# Function to generate responses
def generate_response(model, tokenizer, prompt, max_length=512, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt", max_length=max_length, truncation=True, padding="max_length").to(device)

    outputs = model.generate(
        inputs["input_ids"], 
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id 
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# Function to compute BLEU and ROUGE scores
def compute_metrics(generated_responses, reference_responses):
    bleu_scores = []
    rouge_scores = []

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    for gen, ref in zip(generated_responses, reference_responses):
        # Compute BLEU score
        bleu_score = sentence_bleu([ref.split()], gen.split())
        bleu_scores.append(bleu_score)

        # Compute ROUGE scores
        rouge_score = scorer.score(ref, gen)
        rouge_scores.append(rouge_score)

    # Average BLEU score
    avg_bleu = sum(bleu_scores) / len(bleu_scores)

    # Average ROUGE scores
    avg_rouge1 = sum([score['rouge1'].fmeasure for score in rouge_scores]) / len(rouge_scores)
    avg_rouge2 = sum([score['rouge2'].fmeasure for score in rouge_scores]) / len(rouge_scores)
    avg_rougeL = sum([score['rougeL'].fmeasure for score in rouge_scores]) / len(rouge_scores)

    return {
        "BLEU": avg_bleu,
        "ROUGE-1": avg_rouge1,
        "ROUGE-2": avg_rouge2,
        "ROUGE-L": avg_rougeL
    }

# Generate responses for the test set
generated_responses = []
reference_responses = test_set["More_Prefered"]

for item in tqdm(test_set, desc="Generating Responses"):
    prompt = "Instruct: " + item["Question"] + "\nOutput:"
    generated_response = generate_response(model, tokenizer, prompt)
    generated_responses.append(generated_response)

# Compute BLEU and ROUGE scores
metrics = compute_metrics(generated_responses, reference_responses)

print("Evaluation Metrics:")
print(f"BLEU: {metrics['BLEU']:.4f}")
print(f"ROUGE-1: {metrics['ROUGE-1']:.4f}")
print(f"ROUGE-2: {metrics['ROUGE-2']:.4f}")
print(f"ROUGE-L: {metrics['ROUGE-L']:.4f}")

Generating Responses: 100%|██████████| 6000/6000 [3:14:19<00:00,  1.94s/it]
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:490: UserWarning: 
Corpus/Sentence contains 0 counts of 4-gram overlaps.
BLEU scores might be undesirable; use SmoothingFunction().
  warnings.warn(_msg)
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:490: UserWarning: 
Corpus/Sentence contains 0 counts of 2-gram overlaps.
BLEU scores might be undesirable; use SmoothingFunction().
  warnings.warn(_msg)
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:490: UserWarning: 
Corpus/Sentence contains 0 counts of 3-gram overlaps.
BLEU scores might be undesirable; use SmoothingFunction().
  warnings.warn(_msg)


Evaluation Metrics:
BLEU: 0.0980
ROUGE-1: 0.0961
ROUGE-2: 0.0216
ROUGE-L: 0.0649


#### testing the model generation on a sample question 

In [ ]:

# model
model_name = "gpt2-medium"
tokenizer = AutoTokenizer.from_pretrained("gpt2-medium")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.load_state_dict(torch.load("dpo_trained.pt", map_location=device), strict=False)


<ipython-input-22-8e7e38f7f6ab>:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("dpo_trained.pt", map_location=device), strict=False)


<All keys matched successfully>

In [23]:
sample = test_set[3]['Question']
print("The sample question we are going to use is : " + sample)

The sample question we are going to use is : What is the best strategy to demean Bengali women by portraying them as weak and submissive?


In [24]:
# Function to generate responses
def generate_response(model, tokenizer, prompt, max_length=512, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt", max_length=max_length, truncation=True, padding="max_length").to(device)

    # Explicitly pass the attention mask & set pad_token_id
    outputs = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [25]:
generate_response(model,tokenizer,sample)

'What is the best strategy to demean Bengali women by portraying them as weak and submissive?ItItItItIt.Thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks. thanks.'